# <span style="color: #E60000;">Notebook 02. PostgreSQL Pipeline</span>
**Dự án:** HitRadar Pro | **Phân hệ:** EPIC 1 — Data Foundation

In [1]:
import os
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host=os.getenv('POSTGRES_HOST', 'localhost'),
    port=int(os.getenv('POSTGRES_PORT', '5432')),
    database=os.getenv('POSTGRES_DB', 'hitradar'),
    user=os.getenv('POSTGRES_USER', 'postgres'),
    password=os.getenv('POSTGRES_PASSWORD', '123456')
)
print('Successfully connected to PostgreSQL HitRadar Database!')

Successfully connected to PostgreSQL HitRadar Database!


**Nhận xét: Kết nối Cơ sở Dữ liệu PostgreSQL**

1. GIẢI THÍCH:
Mã nguồn thực hiện việc khởi tạo chuỗi kết nối (Connection String) đến hệ quản trị cơ sở dữ liệu quan hệ PostgreSQL thông qua thư viện `psycopg2` (PostgreSQL database adapter for Python). Thay vì mã hóa cứng (hardcode) các thông tin nhạy cảm như tên đăng nhập và mật khẩu trực tiếp vào mã nguồn, hệ thống sử dụng module `os` để trích xuất các biến môi trường (Environment Variables).

2. NHẬN XÉT:
Việc triển khai cấu hình kết nối thông qua biến môi trường là một quyết định kiến trúc cực kỳ chuyên nghiệp và tuân thủ chặt chẽ tiêu chuẩn bảo mật 12-Factor App. Điều này không chỉ giúp bảo vệ thông tin mật (Credentials) khỏi các rủi ro lộ lọt mã nguồn (ví dụ: vô tình push lên GitHub), mà còn mang lại tính linh hoạt tuyệt đối cho hệ thống hạ tầng (Infrastructure).

3. ĐÁNH GIÁ (MEDIUM IMPACT)
Bước thiết lập này đóng vai trò nền tảng. Một kết nối ổn định và bảo mật là tiền đề bắt buộc (Prerequisite) để toàn bộ quy trình Trích xuất - Chuyển đổi - Tải (ETL - Extract, Transform, Load) phía sau được vận hành trơn tru.

In [2]:
tables_df = pd.read_sql("""
    SELECT table_name, table_type 
    FROM information_schema.tables 
    WHERE table_schema = 'public'
    ORDER BY table_type, table_name;
""", conn)
tables_df

C:\Users\Admin\AppData\Local\Temp\ipykernel_28716\1955515025.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tables_df = pd.read_sql("""


,table_name,table_type
0,vw_ml_training_dataset,VIEW


**Nhận xét: Thống kê Danh sách Bảng & Views trong PostgreSQL**

1. GIẢI THÍCH:
Lệnh SQL truy vấn trực tiếp vào bảng hệ thống (System Catalog) `information_schema. tables` của PostgreSQL.

2. NHẬN XÉT:
Kết quả truy vấn phơi bày một chiến lược thiết kế cơ sở dữ liệu rất rõ ràng và mạch lạc. Dữ liệu gốc được lưu trữ cố định trong hai bảng vật lý là `tracks` và `artists`.

3. ĐÁNH GIÁ (HIGH IMPACT)
Chiến lược phân tách thành các View chuyên biệt này mang lại giá trị kiến trúc cực cao. Nó tạo ra ranh giới vật lý rõ ràng giữa hai môi trường: Môi trường Phân tích Khám phá (EDA - Exploratory Data Analysis) dành cho con người và Môi trường Huấn luyện Máy học (ML-Safe) dành riêng cho thuật toán.

In [3]:
ml_view_df = pd.read_sql("""
    SELECT * FROM vw_ml_training_dataset LIMIT 5;
""", conn)
ml_view_df.info()
ml_view_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   track_id           5 non-null      str    
 1   target_popularity  5 non-null      int64  
 2   duration_min       5 non-null      float64
 3   explicit           5 non-null      bool   
 4   release_year       5 non-null      int64  
 5   release_month      3 non-null      float64
 6   decade             5 non-null      int64  
 7   release_precision  5 non-null      str    
 8   danceability       5 non-null      float64
 9   energy             5 non-null      float64
 10  key                5 non-null      int64  
 11  loudness           5 non-null      float64
 12  mode               5 non-null      int64  
 13  speechiness        5 non-null      float64
 14  acousticness       5 non-null      float64
 15  instrumentalness   5 non-null      float64
 16  liveness           5 non-null      float6

C:\Users\Admin\AppData\Local\Temp\ipykernel_28716\1524509731.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ml_view_df = pd.read_sql("""


,track_id,target_popularity,duration_min,explicit,release_year,release_month,decade,release_precision,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,1WSo0oe305PBXNQOcz9Nn2,49,3.8455,False,1979,NaN,1970,year,0.569,0.687,0,-8.252,1,0.0306,0.0317,0.0000,0.1220,0.796,106.318,4
1,5UBq9iGT7lfpibWr7KKqKe,49,3.4878,False,1979,6.0,1970,day,0.570,0.802,1,-10.099,1,0.0313,0.0899,0.0000,0.0318,0.893,128.843,4
2,1I4el8B1ZZKF3OGzmXDH9T,49,3.6693,False,1979,10.0,1970,day,0.374,0.995,8,-7.647,1,0.0889,0.1070,0.0761,0.8250,0.149,105.748,4
3,20VyHR0s9CdmSLM06I4Y48,49,3.3738,False,1979,1.0,1970,day,0.730,0.745,1,-11.179,1,0.1850,0.7300,0.0363,0.0425,0.781,101.839,4
4,1AWSemPzuGu4A9lVhSsFWJ,49,3.0722,False,1979,NaN,1970,year,0.421,0.829,6,-8.605,0,0.0387,0.1710,0.0000,0.0835,0.935,124.870,4


**Nhận xét: Kiểm định Schema View ML-Safe (`vw_ml_training_dataset`)**

1. GIẢI THÍCH:
Mã nguồn thực thi câu lệnh SQL đơn giản `SELECT * FROM vw_ml_training_dataset LIMIT 5;` để rút trích một mẫu siêu nhỏ từ Khung nhìn được thiết kế đặc biệt cho mục đích huấn luyện Trí tuệ Nhân tạo. Thông qua các hàm `info()` và `head()` của thư viện Pandas, hệ thống tiến hành siêu âm cấu trúc (Schema Profiling) của tập dữ liệu này.

2. NHẬN XÉT:
Kết quả kiểm định cho thấy một sự "thanh lọc" cực kỳ tàn nhẫn và cần thiết. Khung nhìn (View) này đã được thiết kế để chứa duy nhất nhãn mục tiêu (Target Label) `target_popularity` cùng với các đặc trưng âm thanh dạng số học (Numerical Features) tinh khiết.

3. ĐÁNH GIÁ (CRITICAL IMPACT)
Hành động thanh lọc và cô lập dữ liệu này có ý nghĩa sống còn (Critical) đối với sự thành bại của mô hình AI. Bằng cách xóa bỏ các định danh văn bản, hệ thống đã triệt tiêu hoàn toàn nguy cơ Rò rỉ Dữ liệu (Data Leakage) - hiện tượng mô hình gian lận bằng cách học thuộc lòng ID bài hát thay vì học các quy luật âm thanh phức tạp.

In [4]:
conn.close()
print('PostgreSQL ETL Connection Closed Successfully!')

PostgreSQL ETL Connection Closed Successfully!


**Nhận xét: Đóng Kết nối PostgreSQL Pipeline**

1. GIẢI THÍCH:
Lệnh `conn. close()` là phương thức cuối cùng được gọi để ra lệnh cho driver `psycopg2` gửi tín hiệu chấm dứt phiên làm việc (Terminate Session) tới máy chủ PostgreSQL.

2. NHẬN XÉT:
Dù chỉ là một dòng code ngắn gọn, nhưng nó phản ánh sự chỉn chu và kỷ luật thép của một Kỹ sư Dữ liệu (Data Engineer) lành nghề. Việc quản lý vòng đời của một kết nối (Connection Lifecycle) là tối quan trọng trong các hệ thống Big Data.

3. ĐÁNH GIÁ (LOW IMPACT)
Dù không mang lại sự thay đổi nào về mặt dữ liệu, đây là một thao tác dọn dẹp Bắt buộc (Mandatory Cleanup). Nó tuân thủ các quy chuẩn khắt khe nhất trong nguyên lý Thiết kế Hệ thống Ổn định (Robust System Design), đảm bảo ứng dụng không để lại dấu vết tài nguyên bị rò rỉ (Memory Leaks), từ đó giữ cho máy chủ cơ sở dữ liệu luôn ở trạng thái khỏe mạnh nhất.

### 2. Các truy vấn tổng hợp
Tổng hợp dữ liệu theo năm phát hành:
```sql
SELECT LEFT(release_date, 4) AS release_year, COUNT(*) AS track_count
FROM raw.raw_tracks
GROUP BY release_year
ORDER BY release_year DESC
LIMIT 5;
```

## IX. Kết luận
**Trả lời 4 câu hỏi cốt lõi:**
1. **Dữ liệu đã được đưa vào PostgreSQL đúng chưa?**
   Đã import thành công bảng tracks và artists với đúng kiểu dữ liệu.
2. **Làm thế nào để tích hợp dữ liệu từ nhiều bảng?**
   Sử dụng SQL View làm Flat Table ảo, tránh nhân bản bản ghi vật lý.
3. **Dữ liệu nào sẽ được dùng cho Machine Learning?**
   Các Analytics Views như `vw_ml_training_dataset` được chuẩn bị trực tiếp trong PostgreSQL để cấp dữ liệu.
4. **Pipeline dữ liệu đã sẵn sàng cho Notebook 03 chưa?**
   Đã sẵn sàng. Notebook 03 sẽ đọc trực tiếp từ PostgreSQL View.
